# OralVerse — MeshSegNet Training (Kaggle)

**Platform**: Kaggle Notebooks — free P100 GPU, 30 GPU hrs/week

**Setup**:
1. New Notebook → Settings (right panel) → Accelerator: **GPU P100**
2. Settings → Internet: **On** (needed to clone repo and download dataset)
3. Fill in your GitHub repo URL in cell 2
4. Run All (`Run` menu → `Run All`)
5. Come back in ~5 hours
6. Download `.pt` files from the **Output** tab on the right

**Expected output**: `meshsegnet_upper_best.pt` + `meshsegnet_lower_best.pt`  
Place them in `ai/orthodontics/segmentation/meshsegnet/checkpoints/` locally.

## 1. Check GPU

In [ ]:
import torch

print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu = torch.cuda.get_device_properties(0)
    print(f"GPU     : {gpu.name}")
    print(f"VRAM    : {gpu.total_memory / 1e9:.1f} GB")
else:
    raise RuntimeError(
        "No GPU detected.\n"
        "Go to: Settings (right panel) → Accelerator → GPU P100"
    )

## 2. Clone repo

In [ ]:
import os
from pathlib import Path

REPO_URL  = "https://github.com/sh1v3n/OralVerse-CAD.git"  # ← your repo
BRANCH    = "claude/features"
WORK_DIR  = Path("/kaggle/working/OralVerse-CAD")

if not WORK_DIR.exists():
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {WORK_DIR}
else:
    !cd {WORK_DIR} && git pull

os.chdir(WORK_DIR)
print(f"Working directory: {Path.cwd()}")

## 3. Install dependencies

In [ ]:
REQ = WORK_DIR / "ai/orthodontics/segmentation/meshsegnet/requirements.txt"
!pip install -q -r {REQ}
print("Dependencies installed.")

## 4. Download dataset

`DOWNLOAD_ALL = False` fetches only `01.zip` (~2 GB, ~450 scans) — enough for a good model in ~5 hrs.  
Set `True` for the full 1800-scan dataset (~8 GB, ~4× longer).

In [ ]:
DOWNLOAD_ALL = False

RAW_DIR  = WORK_DIR / "ai/orthodontics/segmentation/meshsegnet/raw_data"
DATA_DIR = WORK_DIR / "ai/orthodontics/segmentation/meshsegnet/data"
CKPT_DIR = WORK_DIR / "ai/orthodontics/segmentation/meshsegnet/checkpoints"

for d in [RAW_DIR, DATA_DIR, CKPT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

BASE  = "https://zenodo.org/record/7151927/files"
PARTS = ["01", "02", "03", "04"] if DOWNLOAD_ALL else ["01"]

for part in PARTS:
    dest = RAW_DIR / f"{part}.zip"
    if not dest.exists():
        print(f"Downloading {part}.zip ...")
        !curl -L --progress-bar -o {dest} "{BASE}/{part}.zip?download=1"
    else:
        print(f"[skip] {part}.zip already present")
    print(f"Extracting {part}.zip ...")
    !unzip -q -o {dest} -d {RAW_DIR}

n_obj = len(list(RAW_DIR.rglob("*.obj")))
print(f"\nDataset ready: {n_obj} OBJ files")

## 5. Preprocess scans → .npz

In [ ]:
!python -m ai.orthodontics.segmentation.meshsegnet.preprocess \
    --data_dir {RAW_DIR} \
    --out_dir  {DATA_DIR}

n_npz = len(list(DATA_DIR.rglob("*.npz")))
print(f"\nPreprocessed: {n_npz} .npz files")

## 6. Train — Upper arch

In [ ]:
EPOCHS    = 50     # 50 = good model; 100 = best quality (doubles time)
MAX_FACES = 12000  # reduce to 8000 if you hit OOM

!python -m ai.orthodontics.segmentation.meshsegnet.train \
    --data_dir  {DATA_DIR} \
    --arch      upper \
    --out_dir   {CKPT_DIR} \
    --epochs    {EPOCHS} \
    --max_faces {MAX_FACES}

## 7. Train — Lower arch

In [ ]:
!python -m ai.orthodontics.segmentation.meshsegnet.train \
    --data_dir  {DATA_DIR} \
    --arch      lower \
    --out_dir   {CKPT_DIR} \
    --epochs    {EPOCHS} \
    --max_faces {MAX_FACES}

## 8. Evaluate

In [ ]:
for arch in ["upper", "lower"]:
    ckpt = CKPT_DIR / f"meshsegnet_{arch}_best.pt"
    if ckpt.exists():
        !python -m ai.orthodontics.segmentation.meshsegnet.evaluate \
            --checkpoint {ckpt} \
            --data_dir   {DATA_DIR} \
            --split      test
    else:
        print(f"No checkpoint found for {arch} — check training output above.")

## 9. Summary

Your checkpoints are now in the **Output** tab on the right side of Kaggle.  
Download them and place locally at:
```
ai/orthodontics/segmentation/meshsegnet/checkpoints/
  meshsegnet_upper_best.pt
  meshsegnet_lower_best.pt
```

Then verify locally:
```bash
python -m ai.orthodontics.segmentation.meshsegnet.export_model \
    --checkpoint ai/orthodontics/segmentation/meshsegnet/checkpoints/meshsegnet_upper_best.pt
```

Then activate:
```bash
export ORALVERSE_SEGMENTER=meshsegnet
export MESHSEGNET_WEIGHTS=/path/to/checkpoints/meshsegnet_upper_best.pt
```

In [ ]:
checkpoints = list(CKPT_DIR.glob("*.pt"))
if not checkpoints:
    print("No checkpoints found — check training logs above for errors.")
else:
    print(f"Checkpoints ({len(checkpoints)}):")
    for c in sorted(checkpoints):
        print(f"  {c.name}  ({c.stat().st_size / 1e6:.1f} MB)")
    print("\nDownload from the Output tab on the right.")